In [9]:
import warnings
warnings.filterwarnings('ignore')
import glob
import numpy as np
import pandas as pd
from astropy.io import fits as pyfits
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import matplotlib as mpl
# use precise epoch
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00'
try: mdates.set_epoch('1970-01-01T00:00:00')
except: pass

data_dir = './sample_data/orfees'

In [12]:
mydate = '2025-03-25'
year, month, day = mydate.split('-')

#### From Shilpi Bhunia
> Originally prepared by Shane Maloney and Pearse Murphy

In [3]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import time, os
from matplotlib import dates
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
from datetime import datetime
from matplotlib import dates
import matplotlib as mpl
from astropy.io import fits
import astropy.units as u
from sunpy.net import Fido, attrs as a
from astropy.time import Time
from matplotlib.colors import LogNorm

In [18]:
files = glob.glob(f'{data_dir}/int_orf{year}{month}{day}_*.fts')
orfees = fits.open(files[0])
orfees.info()

Filename: ./sample_data/orfees/int_orf20250325_053100_0.1.fts
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      16   ()      
  1  FREQUENCIES    1 BinTableHDU     37   1R x 11C   [1J, 1J, 431E, 1J, 215E, 1J, 164E, 1J, 86E, 1J, 102E]   
  2  SPECTRA       1 BinTableHDU     78   358292R x 15C   [1J, 431E, 431E, 1J, 215E, 215E, 1J, 164E, 164E, 1J, 86E, 86E, 1J, 102E, 102E]   


In [21]:
orfees_i = np.hstack([orfees[2].data[f'STOKESI_B{i}'] for i in range(1, 6)]).T
print(orfees_i.shape)

data = orfees_i.T
print(data.shape)

(998, 358292)
(358292, 998)


In [23]:
orfees_time_str = orfees[0].header['DATE-OBS']
orfees_times = Time(orfees_time_str) + (orfees[2].data['TIME_B1']/1000)*u.s # times are not the same for all sub spectra
orfees_freqs = np.hstack([orfees[1].data[f'FREQ_B{i}'] for i in range(1, 6)])*u.MHz

df = pd.DataFrame(data=data, index=orfees_times, columns=orfees_freqs.value.reshape(-1))
df.index.name   = 'Time'
df.columns.name = 'Frequency'
df.index = [t.datetime for t in df.index]

# downsample to 1s
df_orfees_1s = df.resample('1S').mean()

# remove const bkg
df_orfees_1s_nobkg = df_orfees_1s - np.tile(np.median(df_orfees_1s,0), (df_orfees_1s.shape[0],1))
df_orfees_1s_nobkg.head()

Frequency,144.130005,144.520004,144.910004,145.300003,145.690002,146.960007,148.089996,148.429993,148.820007,149.210007,...,990.729980,992.289978,993.859985,995.419983,996.979980,998.539978,1000.109985,1001.669983,1003.229980,1004.700012
2025-03-25 06:56:06,3.178669,2.185333,-2.334164,-1.308334,-0.298920,-1.542000,-0.910561,-0.761501,-1.109169,-1.384335,...,-14.256870,-15.188087,-15.542496,-16.903656,-16.618740,-14.645596,-15.164291,-15.652504,-17.251881,-1.762856
2025-03-25 06:56:07,1.192001,0.000000,-2.044250,-0.870003,-1.214252,-1.380497,-0.844334,-0.747002,-0.374500,-1.074753,...,33.049374,67.738739,32.024883,-5.054573,-8.779217,-10.950596,-12.618874,-13.423752,-15.479378,-1.942856
2025-03-25 06:56:08,3.576000,1.191998,-1.674000,-1.085499,-0.519253,-1.462498,-0.645672,-0.860252,-0.373497,-0.963753,...,-14.387806,-14.905334,-15.542496,-16.088753,-16.052471,-14.645596,-15.712624,-16.105000,-17.155006,-1.959282
2025-03-25 06:56:09,3.278000,1.042995,-1.783497,-0.743500,3.222252,-1.459496,-0.695335,-1.645500,-0.934254,-0.407253,...,-12.893810,-14.771275,-15.493870,-15.301689,-15.506531,-14.645596,-15.898750,-15.837502,-17.251884,-1.959282
2025-03-25 06:56:10,2.384003,1.639000,-1.468002,-1.183502,2.886005,-1.623497,-1.241669,-2.168999,-1.645500,-0.926254,...,-13.600307,-13.824211,-14.523808,-12.463257,-12.316597,-8.003036,-14.447437,-16.116879,-17.058128,-1.959282


In [ ]:
start_time = '2025-03-25T12:21:00'
end_time   = '2025-03-25T13:12:00'

fig = plt.figure(figsize=[10,6])
ax = fig.add_subplot(111)
pc = ax.pcolormesh(df_orfees_1s_nobkg.index, df_orfees_1s_nobkg.columns, df_orfees_1s_nobkg.T,
                   # vmin=0,
                   # norm=LogNorm(vmin=1e1, vmax=1e2),
                   vmin=np.nanpercentile(df_orfees_1s_nobkg, 50),
                   vmax=np.nanpercentile(df_orfees_1s_nobkg, 97.5),
                   # shading='nearest',
                   cmap='Spectral_r')
fig.colorbar(pc, ax=ax, pad=0.02, label=orfees[2].header['BUNIT'])
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.set_yscale('log')
ax.set_ylim(ax.get_ylim()[::-1])
custom_ticks = [200, 300, 400, 500, 600, 700, 800, 900, 1000]
ax.set_yticks(custom_ticks)
ax.get_yaxis().set_major_formatter(ticker.FormatStrFormatter('%d'))
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlim(left=pd.Timestamp(start_time), right=pd.Timestamp(end_time))
fig.tight_layout()
plt.show()